In [1]:
# Import project modules and the trained FairCredit model.

import sys
from pathlib import Path
import joblib

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.affordability_engine import assess_affordability
from src.simulator import (
    predict_credit_risk,
    simulate_scenario
)

model = joblib.load(
    "../models/faircredit_risk_model.pkl"
)

feature_names = joblib.load(
    "../models/faircredit_model_features.pkl"
)

In [2]:
# Create a realistic example profile for testing the simulator.

profile = {
    "monthly_gross_income": 12_000,
    "statutory_deductions": 1_500,
    "monthly_net_income": 10_500,

    "declared_living_expenses": 4_000,
    "living_expenses_used": 4_000,

    "existing_debt_repayments": 2_500,
    "maintenance_obligations": 500,

    "requested_credit_amount": 20_000,
    "loan_term_months": 12,
    "proposed_installment": 2_000,

    "number_existing_credit_accounts": 3,
    "missed_payments_12m": 2,
    "arrears_amount": 1_000,
    "defaults_history": 0,

    "employment_months": 18,
    "income_variability_pct": 10,

    "savings_amount": 2_000,
    "credit_utilisation_pct": 70
}

In [3]:
# Calculate the user's current FairCredit risk estimate.

current_risk = predict_credit_risk(
    profile,
    model,
    feature_names
)

current_risk

{'risk_probability': np.float64(0.735626288055974),
 'readiness_score': 26,
 'risk_level': 'Higher Risk'}

In [4]:
# Simulate a scenario in which the consumer reduces existing debt
# repayments and requests a smaller amount of credit.

scenario = simulate_scenario(
    original_profile=profile,
    model=model,
    feature_names=feature_names,
    changes={
        "existing_debt_repayments": 1_500,
        "requested_credit_amount": 12_000,
        "proposed_installment": 1_200
    }
)

print("ORIGINAL READINESS")
print(
    f"{current_risk['readiness_score']}/100"
)

print("\nSIMULATED READINESS")
print(
    f"{scenario['risk']['readiness_score']}/100"
)

print("\nSIMULATED AFFORDABILITY")
print(
    "PASS"
    if scenario["affordability"].affordability_pass
    else "FAIL"
)

print(
    scenario["affordability"].status
)

ORIGINAL READINESS
26/100

SIMULATED READINESS
44/100

SIMULATED AFFORDABILITY
PASS
Comfortable affordability
